<a id="derivative"></a>
$$
\text{导数 (Derivative): 函数在某一点的瞬时变化率} \\
f'(x) = \frac{df}{dx} = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h} \\ \quad \\
\text{几何意义: } \frac{f(x+h) - f(x)}{h} \text{ 是割线 (secant) 的斜率, } h \to 0 \text{ 时割线变成切线 (tangent), 导数就是切线斜率} \\ \quad \\
\text{用定义推导常见函数的导数:} \\ \quad \\
1. \ (x^n)' \text{: 二项式展开 } (x + h)^n = x^n + n x^{n-1} h + \binom{n}{2} x^{n-2} h^2 + \cdots + h^n \\
\frac{(x+h)^n - x^n}{h} = n x^{n-1} + \underbrace{\binom{n}{2} x^{n-2} h + \cdots + h^{n-1}}_{\text{都带 h, } h \to 0 \text{ 时为 0}} \to n x^{n-1} \\ \quad \\
2. \ (e^x)' \text{: } \frac{e^{x+h} - e^x}{h} = e^x \cdot \frac{e^h - 1}{h} \to e^x \quad (\text{e 的定义就是让 } \lim_{h \to 0} \frac{e^h - 1}{h} = 1 \text{ 的那个底数}) \\ \quad \\
3. \ (\sin x)' \text{: } \frac{\sin(x+h) - \sin x}{h} = \frac{\sin x \cos h + \cos x \sin h - \sin x}{h} = \sin x \cdot \frac{\cos h - 1}{h} + \cos x \cdot \frac{\sin h}{h} \\
\text{因为 } \lim_{h \to 0} \frac{\sin h}{h} = 1, \ \lim_{h \to 0} \frac{\cos h - 1}{h} = 0 \text{, 所以 } (\sin x)' = \cos x \text{, 同理 } (\cos x)' = -\sin x
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad, solve_ivp

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

# 用定义直接算: h 越小, 割线斜率越接近真实导数
x0 = 1.0
for h in [1e-1, 1e-2, 1e-3, 1e-4]:
    print(f'h = {h:.0e}: (x^3)\' ≈ {((x0 + h)**3 - x0**3) / h:.6f} (真值 3), '
          f'(e^x)\' ≈ {(np.exp(x0 + h) - np.exp(x0)) / h:.6f} (真值 {np.exp(x0):.6f}), '
          f'(sin x)\' ≈ {(np.sin(x0 + h) - np.sin(x0)) / h:.6f} (真值 {np.cos(x0):.6f})')

# 两个关键极限
h_values = np.array([1e-1, 1e-3, 1e-5])
print(f'(e^h - 1) / h = {(np.exp(h_values) - 1) / h_values}')
print(f'sin(h) / h = {np.sin(h_values) / h_values}')

# 割线 -> 切线
x = np.linspace(-0.5, 2.5, 200)
plt.figure(figsize=(4, 3))
plt.plot(x, x**2, label='f(x) = x^2')
for h in [1.0, 0.5, 0.1]:
    slope = ((x0 + h)**2 - x0**2) / h
    plt.plot(x, x0**2 + slope * (x - x0), '--', label=f'secant h={h}, slope={slope:.2f}')
plt.plot(x, x0**2 + 2 * x0 * (x - x0), 'k-', label='tangent slope=2')
plt.ylim(-1, 5)
plot_nicely()
plt.show()

<a id="derivative-rules"></a>
$$
\text{求导法则: 全部可以由导数定义推出} \\ \quad \\
\text{1. 乘积法则 (Product Rule): } (fg)' = f'g + fg' \\
f(x+h)g(x+h) - f(x)g(x) = \underbrace{[f(x+h) - f(x)] g(x+h)}_{\text{加一项}} + \underbrace{f(x) [g(x+h) - g(x)]}_{\text{减一项}} \\
\text{两边除以 h, 令 } h \to 0 \text{: } (fg)' = f'(x) g(x) + f(x) g'(x) \\ \quad \\
\text{2. 倒数: } \left(\frac{1}{g}\right)' = \lim_{h \to 0} \frac{1}{h}\left(\frac{1}{g(x+h)} - \frac{1}{g(x)}\right) = \lim_{h \to 0} \frac{-[g(x+h) - g(x)]}{h \, g(x+h) g(x)} = -\frac{g'}{g^2} \\
\text{商法则 (Quotient Rule): } \left(\frac{f}{g}\right)' = \left(f \cdot \frac{1}{g}\right)' = \frac{f'}{g} - \frac{f g'}{g^2} = \frac{f'g - fg'}{g^2} \\ \quad \\
\text{3. 链式法则 (Chain Rule): } y = f(u), \ u = g(x) \implies \frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} = f'(g(x)) \, g'(x) \\
\text{记 } \Delta u = g(x+h) - g(x) \text{, 则 } \frac{f(g(x+h)) - f(g(x))}{h} = \frac{f(u + \Delta u) - f(u)}{\Delta u} \cdot \frac{\Delta u}{h} \\
h \to 0 \text{ 时 } \Delta u \to 0 \text{, 第一项} \to f'(u) \text{, 第二项} \to g'(x) \\ \quad \\
\text{4. 反函数 / 隐函数求导 (Implicit Differentiation): 求 } (\ln x)' \\
e^{\ln x} = x \xrightarrow{\text{两边对 x 求导, 用链式法则}} e^{\ln x} \cdot (\ln x)' = 1 \implies (\ln x)' = \frac{1}{e^{\ln x}} = \frac{1}{x} \\
\text{同理 } (a^x)' = (e^{x \ln a})' = a^x \ln a \text{, } (x^\alpha)' = (e^{\alpha \ln x})' = x^\alpha \cdot \frac{\alpha}{x} = \alpha x^{\alpha - 1} \text{ (任意实数 } \alpha \text{)}
$$

In [ ]:
def central_diff(f, x, h=1e-6):
    # 中心差分, 下面泰勒展开那一节会推导为什么它比定义里的单侧差分更准
    return (f(x + h) - f(x - h)) / (2 * h)

x0 = 0.7
checks = {
    '乘积 (x^2 sin x)\'':        (lambda x: x**2 * np.sin(x),      2 * x0 * np.sin(x0) + x0**2 * np.cos(x0)),
    '商 (sin x / x)\'':          (lambda x: np.sin(x) / x,         (np.cos(x0) * x0 - np.sin(x0)) / x0**2),
    '链式 sin(x^2)\'':           (lambda x: np.sin(x**2),          np.cos(x0**2) * 2 * x0),
    '链式 e^(-x^2/2)\'':         (lambda x: np.exp(-x**2 / 2),     -x0 * np.exp(-x0**2 / 2)),
    '隐函数 (ln x)\'':           (np.log,                          1 / x0),
    '(2^x)\'':                   (lambda x: 2**x,                  2**x0 * np.log(2)),
    '(x^1.5)\'':                 (lambda x: x**1.5,                1.5 * x0**0.5),
}
for name, (f, analytic) in checks.items():
    numeric = central_diff(f, x0)
    print(f'{name:<20} 推导结果 = {analytic:.8f}, 数值差分 = {numeric:.8f}, 一致? {np.isclose(analytic, numeric)}')

<a id="taylor"></a>
$$
\text{泰勒展开 (Taylor Expansion): 用多项式在一点附近逼近函数} \\ \quad \\
\text{推导: 假设 f 可以写成 } f(x) = a_0 + a_1 (x - a) + a_2 (x - a)^2 + a_3 (x - a)^3 + \cdots \\
x = a \text{ 代入: } f(a) = a_0 \\
\text{求一次导再代入 } x = a \text{: } f'(a) = a_1 \\
\text{求两次导: } f''(a) = 2 a_2 \\
\text{求 k 次导: } (x - a)^k \text{ 求 k 次导等于 } k! \text{, 更低次的项求导后变成 0, 更高次的项代入 } x = a \text{ 后为 0} \\
\implies f^{(k)}(a) = k! \, a_k \implies a_k = \frac{f^{(k)}(a)}{k!} \\ \quad \\
f(x) = \sum_{k=0}^{n} \frac{f^{(k)}(a)}{k!} (x - a)^k + R_n(x), \quad R_n(x) = \frac{f^{(n+1)}(\xi)}{(n+1)!} (x - a)^{n+1} \ (\xi \text{ 在 a 和 x 之间, 拉格朗日余项}) \\ \quad \\
\text{常写成步长 h 的形式 (下面数值差分, 数值积分, 解 ODE 都要用):} \\
f(x + h) = f(x) + h f'(x) + \frac{h^2}{2} f''(x) + \frac{h^3}{6} f'''(x) + O(h^4) \\ \quad \\
\text{在 a = 0 处 (麦克劳林 Maclaurin):} \\
e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \cdots, \quad
\sin x = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \cdots, \quad
\cos x = 1 - \frac{x^2}{2!} + \frac{x^4}{4!} - \cdots \\ \quad \\
\text{推论: 欧拉公式 (Euler's Formula), 把 } ix \text{ 代入 } e^x \text{ 的级数, 利用 } i^2 = -1, i^3 = -i, i^4 = 1 \text{:} \\
e^{ix} = 1 + ix - \frac{x^2}{2!} - i\frac{x^3}{3!} + \frac{x^4}{4!} + \cdots = \underbrace{\left(1 - \frac{x^2}{2!} + \frac{x^4}{4!} - \cdots\right)}_{\cos x} + i \underbrace{\left(x - \frac{x^3}{3!} + \cdots\right)}_{\sin x} \\
e^{ix} = \cos x + i \sin x \quad (\text{交流电的相量, 光的波动, 傅里叶变换都建立在这个公式上})
$$

In [ ]:
from math import factorial

def taylor_sin(x, n_terms):
    return sum((-1)**k * x**(2*k + 1) / factorial(2*k + 1) for k in range(n_terms))

x = np.linspace(-2 * np.pi, 2 * np.pi, 400)
plt.figure(figsize=(4, 3))
plt.plot(x, np.sin(x), 'k', linewidth=2, label='sin x')
for n_terms in [1, 2, 4, 7]:
    plt.plot(x, taylor_sin(x, n_terms), '--', label=f'degree {2 * n_terms - 1}')
plt.ylim(-2, 2)
plt.title('Taylor polynomial of sin x')
plot_nicely()
plt.show()

# 余项: 截断在 n 次时, 误差 ~ h^(n+1)
for h in [0.1, 0.05, 0.025]:
    err1 = abs(np.exp(h) - (1 + h))               # 1 次, 误差 ~ h^2 / 2
    err2 = abs(np.exp(h) - (1 + h + h**2 / 2))    # 2 次, 误差 ~ h^3 / 6
    print(f'h = {h:<6}: 1 次误差 = {err1:.3e} (h^2/2 = {h**2/2:.3e}), 2 次误差 = {err2:.3e} (h^3/6 = {h**3/6:.3e})')
print('h 减半时, 1 次误差约变成 1/4, 2 次误差约变成 1/8')

# 欧拉公式
theta = np.linspace(0, 2 * np.pi, 7)
series = sum((1j * theta)**k / factorial(k) for k in range(30))
print(f'级数 e^(i theta) == cos + i sin ? {np.allclose(series, np.cos(theta) + 1j * np.sin(theta))}')
print(f'e^(i pi) + 1 = {np.exp(1j * np.pi) + 1:.2e}')

<a id="numerical-diff"></a>
$$
\text{数值微分 (Numerical Differentiation): 由泰勒展开推导误差阶} \\ \quad \\
f(x + h) = f(x) + h f'(x) + \frac{h^2}{2} f''(x) + \frac{h^3}{6} f'''(x) + \frac{h^4}{24} f''''(x) + \cdots \quad (1) \\
f(x - h) = f(x) - h f'(x) + \frac{h^2}{2} f''(x) - \frac{h^3}{6} f'''(x) + \frac{h^4}{24} f''''(x) - \cdots \quad (2) \\ \quad \\
\text{1. 前向差分 (Forward Difference), 由 (1): } \frac{f(x+h) - f(x)}{h} = f'(x) + \underbrace{\frac{h}{2} f''(x) + \cdots}_{O(h)} \\ \quad \\
\text{2. 中心差分 (Central Difference), (1) - (2): 偶数次项抵消} \\
f(x+h) - f(x-h) = 2h f'(x) + \frac{h^3}{3} f'''(x) + \cdots \implies \frac{f(x+h) - f(x-h)}{2h} = f'(x) + \underbrace{\frac{h^2}{6} f'''(x) + \cdots}_{O(h^2)} \\ \quad \\
\text{3. 二阶导, (1) + (2): 奇数次项抵消} \\
f(x+h) + f(x-h) = 2 f(x) + h^2 f''(x) + \frac{h^4}{12} f''''(x) + \cdots \implies \frac{f(x+h) - 2f(x) + f(x-h)}{h^2} = f''(x) + O(h^2) \\
\text{(物理里热传导方程, 波动方程的有限差分就是用这个公式)} \\ \quad \\
\text{4. h 不能无限小: 浮点数有舍入误差 } \varepsilon \approx 10^{-16} \text{, 分子相减后除以 h, 舍入误差约为 } \frac{\varepsilon}{h} \\
\text{总误差} \approx \frac{h}{2}|f''| + \frac{\varepsilon}{h} \text{ (前向), 对 h 求导令其为 0: 最优 } h \approx \sqrt{\varepsilon} \approx 10^{-8} \\
\text{中心差分: 总误差} \approx \frac{h^2}{6}|f'''| + \frac{\varepsilon}{h} \text{, 最优 } h \approx \varepsilon^{1/3} \approx 10^{-5}
$$

In [ ]:
f, df, d2f = np.exp, np.exp, np.exp
x0 = 1.0
hs = np.logspace(-15, -1, 60)
err_forward = np.abs((f(x0 + hs) - f(x0)) / hs - df(x0))
err_central = np.abs((f(x0 + hs) - f(x0 - hs)) / (2 * hs) - df(x0))
err_second = np.abs((f(x0 + hs) - 2 * f(x0) + f(x0 - hs)) / hs**2 - d2f(x0))

plt.figure(figsize=(4, 3))
plt.loglog(hs, err_forward, label='forward O(h)')
plt.loglog(hs, err_central, label='central O(h^2)')
plt.loglog(hs, err_second, label="second derivative O(h^2)")
plt.ylim(1e-12, 1e2)
plt.xlabel('h')
plt.ylabel('|error|')
plot_nicely()
plt.show()

print(f'前向差分最小误差在 h = {hs[np.argmin(err_forward)]:.1e} (理论 ~1e-8)')
print(f'中心差分最小误差在 h = {hs[np.argmin(err_central)]:.1e} (理论 ~1e-5)')
# 右半段 (h 大) 斜率 = 误差阶; 左半段 (h 小) 斜率 = -1, 舍入误差主导

<a id="partial"></a>
$$
\text{偏导数 (Partial Derivative): 多元函数只让一个变量变化, 其他变量固定时的导数} \\
\frac{\partial f}{\partial x}(x, y) = \lim_{h \to 0} \frac{f(x + h, y) - f(x, y)}{h}, \quad
\frac{\partial f}{\partial y}(x, y) = \lim_{h \to 0} \frac{f(x, y + h) - f(x, y)}{h} \\
\text{求法: 把其他变量当常数, 用一元的求导法则} \\ \quad \\
\text{二阶偏导: } f_{xx} = \frac{\partial^2 f}{\partial x^2}, \ f_{xy} = \frac{\partial}{\partial y}\left(\frac{\partial f}{\partial x}\right) \\
\text{施瓦茨定理 (Schwarz / Clairaut): 二阶偏导连续时 } f_{xy} = f_{yx} \text{ (所以后面的 Hessian 矩阵是对称矩阵)} \\ \quad \\
\text{全微分 (Total Differential): 两个变量同时变化} \\
f(x + \Delta x, y + \Delta y) - f(x, y) = \underbrace{[f(x + \Delta x, y + \Delta y) - f(x, y + \Delta y)]}_{\approx f_x \Delta x} + \underbrace{[f(x, y + \Delta y) - f(x, y)]}_{\approx f_y \Delta y} \\
df = \frac{\partial f}{\partial x} dx + \frac{\partial f}{\partial y} dy \\ \quad \\
\text{多元链式法则: } z = f(x(t), y(t)) \text{, 全微分两边除以 dt} \\
\frac{dz}{dt} = \frac{\partial f}{\partial x} \frac{dx}{dt} + \frac{\partial f}{\partial y} \frac{dy}{dt}
$$

In [ ]:
def f_xy(x, y):
    return x**2 * y + np.sin(x * y)

# 推导结果 (把另一个变量当常数)
fx  = lambda x, y: 2 * x * y + y * np.cos(x * y)
fy  = lambda x, y: x**2 + x * np.cos(x * y)
fxy = lambda x, y: 2 * x + np.cos(x * y) - x * y * np.sin(x * y)   # 对 fx 再对 y 求导 (乘积法则)

x0, y0, h = 0.8, 1.3, 1e-5
fx_num = (f_xy(x0 + h, y0) - f_xy(x0 - h, y0)) / (2 * h)
fy_num = (f_xy(x0, y0 + h) - f_xy(x0, y0 - h)) / (2 * h)
h2 = 1e-4
fxy_num = (f_xy(x0 + h2, y0 + h2) - f_xy(x0 + h2, y0 - h2) - f_xy(x0 - h2, y0 + h2) + f_xy(x0 - h2, y0 - h2)) / (4 * h2**2)
print(f'f_x : 推导 = {fx(x0, y0):.6f}, 数值 = {fx_num:.6f}')
print(f'f_y : 推导 = {fy(x0, y0):.6f}, 数值 = {fy_num:.6f}')
print(f'f_xy: 推导 = {fxy(x0, y0):.6f}, 数值 = {fxy_num:.6f}')

# 施瓦茨: 先对 x 再对 y == 先对 y 再对 x
fyx_from_fy = (fy(x0 + h, y0) - fy(x0 - h, y0)) / (2 * h)
fxy_from_fx = (fx(x0, y0 + h) - fx(x0, y0 - h)) / (2 * h)
print(f'f_xy = {fxy_from_fx:.6f}, f_yx = {fyx_from_fy:.6f}')

# 多元链式法则: 沿曲线 x = cos t, y = sin t
t0 = 0.4
x_t, y_t = np.cos(t0), np.sin(t0)
dz_dt_chain = fx(x_t, y_t) * (-np.sin(t0)) + fy(x_t, y_t) * np.cos(t0)
z = lambda t: f_xy(np.cos(t), np.sin(t))
print(f'dz/dt: 链式法则 = {dz_dt_chain:.6f}, 直接数值求导 = {(z(t0 + h) - z(t0 - h)) / (2 * h):.6f}')

<a id="gradient"></a>
$$
\text{梯度 (Gradient): 所有偏导数组成的向量} \\
\nabla f = \left( \frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, \cdots, \frac{\partial f}{\partial x_n} \right)^T \\ \quad \\
\text{方向导数 (Directional Derivative): 沿单位向量 u 方向的变化率} \\
D_u f(x) = \lim_{t \to 0} \frac{f(x + t u) - f(x)}{t} \\
\text{推导: 令 } g(t) = f(x + tu) = f(x_1 + t u_1, \cdots, x_n + t u_n) \text{, 用多元链式法则} \\
D_u f = g'(0) = \sum_{i} \frac{\partial f}{\partial x_i} \frac{d(x_i + t u_i)}{dt} = \sum_i \frac{\partial f}{\partial x_i} u_i = \nabla f \cdot u = |\nabla f| \cos\theta \\ \quad \\
\text{结论 (梯度下降, 神经网络训练的基础):} \\
1. \ \theta = 0 \text{ 时 } D_u f \text{ 最大: 沿梯度方向函数上升最快, 最大变化率 = } |\nabla f| \\
2. \ \theta = \pi \text{ 时最小: 沿负梯度方向下降最快} \\
3. \ \theta = \pi / 2 \text{ 时为 0: 梯度垂直于等高线 (contour)} \\
\text{(沿等高线 } r(t) \text{ 走, f 不变, } \frac{d}{dt} f(r(t)) = \nabla f \cdot r'(t) = 0 \text{, 所以梯度垂直于等高线的切向量)}
$$

In [ ]:
def f_bowl(x, y):
    return x**2 + 3 * y**2 + x * y

def grad_bowl(x, y):
    return np.array([2 * x + y, 6 * y + x])

p0 = np.array([1.0, 0.5])
g = grad_bowl(*p0)

# 扫描所有方向, 找方向导数最大的方向
angles = np.linspace(0, 2 * np.pi, 3601)
units = np.column_stack([np.cos(angles), np.sin(angles)])
t = 1e-6
directional = (f_bowl(p0[0] + t * units[:, 0], p0[1] + t * units[:, 1]) - f_bowl(*p0)) / t
best = units[np.argmax(directional)]
print(f'梯度 = {g}, |grad| = {np.linalg.norm(g):.6f}')
print(f'数值上方向导数最大的方向 = {best}, 梯度单位向量 = {g / np.linalg.norm(g)}')
print(f'最大方向导数 = {directional.max():.6f} (== |grad|)')
print(f'D_u f == grad . u ? {np.allclose(directional, units @ g, atol=1e-4)}')

xs, ys = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-2, 2, 200))
qx, qy = np.meshgrid(np.linspace(-1.8, 1.8, 10), np.linspace(-1.8, 1.8, 10))
G = grad_bowl(qx, qy)
plt.figure(figsize=(4, 3))
plt.contour(xs, ys, f_bowl(xs, ys), levels=15)
plt.quiver(qx, qy, G[0], G[1], color='r', label='gradient')
plt.gca().set_aspect('equal')
plt.title('gradient is perpendicular to contours')
plot_nicely()
plt.show()

<a id="jacobian"></a>
$$
\text{雅可比矩阵 (Jacobian Matrix): 向量函数 } F: \mathbb{R}^n \to \mathbb{R}^m \text{ 的一阶导数} \\
J = \frac{\partial F}{\partial x} = \begin{bmatrix}
\frac{\partial F_1}{\partial x_1} & \cdots & \frac{\partial F_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial F_m}{\partial x_1} & \cdots & \frac{\partial F_m}{\partial x_n}
\end{bmatrix}, \quad J_{ij} = \frac{\partial F_i}{\partial x_j} \\ \quad \\
\text{由全微分: 每一个分量 } dF_i = \sum_j \frac{\partial F_i}{\partial x_j} dx_j \implies F(x + h) \approx F(x) + J h \text{ (局部线性近似)} \\ \quad \\
\text{链式法则的矩阵形式: } F(G(x)) \approx F(G(x) + J_G h) \approx F(G(x)) + J_F J_G h \implies J_{F \circ G} = J_F \, J_G \\
\text{(神经网络的反向传播就是一串 Jacobian 相乘)} \\ \quad \\
\text{例: 极坐标 } (r, \theta) \to (x, y) = (r \cos\theta, r \sin\theta) \\
J = \begin{bmatrix} \cos\theta & -r \sin\theta \\ \sin\theta & r \cos\theta \end{bmatrix}, \quad \det J = r \cos^2\theta + r \sin^2\theta = r \\
\text{det J 的意义: 面积缩放倍数 (见 } \text{线性代数行列式), 重积分换元时 } dx\,dy = |\det J| \, dr\,d\theta = r \, dr\,d\theta \\ \quad \\
\text{黑塞矩阵 (Hessian Matrix): 标量函数的二阶导数 = 梯度的 Jacobian} \\
H = \nabla^2 f, \quad H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j} \text{, 由施瓦茨定理 } H_{ij} = H_{ji} \text{, H 是对称矩阵}
$$

数学基础: [行列式与面积缩放](linear_algebra.ipynb#determinant) · [矩阵求导 (Matrix Calculus)](linear_algebra.ipynb#matrix-calculus)

In [ ]:
def numerical_jacobian(F, x, h=1e-6):
    x = np.asarray(x, dtype=float)
    Fx = np.asarray(F(x))
    J = np.zeros((Fx.size, x.size))
    for j in range(x.size):
        e = np.zeros_like(x); e[j] = h
        J[:, j] = (np.asarray(F(x + e)) - np.asarray(F(x - e))) / (2 * h)   # 第 j 列: 对 x_j 的偏导
    return J

polar = lambda p: np.array([p[0] * np.cos(p[1]), p[0] * np.sin(p[1])])
r0, th0 = 2.0, 0.6
J_num = numerical_jacobian(polar, [r0, th0])
J_analytic = np.array([[np.cos(th0), -r0 * np.sin(th0)], [np.sin(th0), r0 * np.cos(th0)]])
print(f'Jacobian 数值 = \n{J_num}\n推导 == 数值? {np.allclose(J_num, J_analytic)}, det J = {np.linalg.det(J_num):.6f} (== r = {r0})')

# 链式法则 J_{F o G} = J_F J_G
G = lambda p: np.array([p[0]**2 + p[1], np.sin(p[0] * p[1])])
F = lambda u: np.array([u[0] * u[1], np.exp(u[0]), u[0] - u[1]**2])   # R^2 -> R^3
p = np.array([0.5, 1.2])
J_composite = numerical_jacobian(lambda q: F(G(q)), p)
J_product = numerical_jacobian(F, G(p)) @ numerical_jacobian(G, p)
print(f'J_(F o G) shape = {J_composite.shape}, == J_F J_G ? {np.allclose(J_composite, J_product, atol=1e-6)}')

# Hessian = 梯度的 Jacobian
grad_f = lambda q: grad_bowl(q[0], q[1])
H = numerical_jacobian(grad_f, [0.3, -0.2])
print(f'f = x^2 + 3y^2 + xy 的 Hessian = \n{H}\n对称? {np.allclose(H, H.T)}')

<a id="multivariate-taylor"></a>
$$
\text{多元泰勒展开: 转化成一元的情况} \\
\text{令 } g(t) = f(x + t h) \text{, 一元泰勒: } g(1) = g(0) + g'(0) + \frac{1}{2} g''(0) + \cdots \\
g'(t) = \sum_i \frac{\partial f}{\partial x_i}(x + th) \, h_i = \nabla f(x + th)^T h \quad (\text{方向导数}) \\
g''(t) = \sum_i \sum_j \frac{\partial^2 f}{\partial x_i \partial x_j}(x + th) \, h_i h_j = h^T H(x + th) \, h \\ \quad \\
f(x + h) = f(x) + \nabla f(x)^T h + \frac{1}{2} h^T H(x) \, h + O(|h|^3) \\ \quad \\
\text{这一个公式推出了后面的: 极值的二阶条件, 梯度下降 (只用一阶项), 牛顿法 (用到二阶项)}
$$

In [ ]:
def f_test(q):
    x, y = q
    return np.exp(x) * np.sin(y) + x**2 * y

def grad_test(q):
    x, y = q
    return np.array([np.exp(x) * np.sin(y) + 2 * x * y, np.exp(x) * np.cos(y) + x**2])

def hess_test(q):
    x, y = q
    return np.array([[np.exp(x) * np.sin(y) + 2 * y, np.exp(x) * np.cos(y) + 2 * x],
                     [np.exp(x) * np.cos(y) + 2 * x, -np.exp(x) * np.sin(y)]])

q0 = np.array([0.3, 0.8])
direction = np.array([0.6, -0.8])
print(f'Hessian 推导 == 数值? {np.allclose(hess_test(q0), numerical_jacobian(grad_test, q0), atol=1e-6)}')
for s in [0.1, 0.05, 0.025]:
    h = s * direction
    first = f_test(q0) + grad_test(q0) @ h
    second = first + 0.5 * h @ hess_test(q0) @ h
    print(f'|h| = {s:<6}: 一阶近似误差 = {abs(f_test(q0 + h) - first):.3e}, 二阶近似误差 = {abs(f_test(q0 + h) - second):.3e}')
print('|h| 减半: 一阶误差约 1/4 (O(h^2)), 二阶误差约 1/8 (O(h^3))')

<a id="extremum"></a>
$$
\text{极值 (Extremum)} \\ \quad \\
\text{一元: 由泰勒展开 } f(x_0 + h) = f(x_0) + f'(x_0) h + \frac{1}{2} f''(x_0) h^2 + O(h^3) \\
\text{1. 一阶必要条件: 若 } f'(x_0) \neq 0 \text{, h 很小时 } f'(x_0) h \text{ 主导, 取 h 的正负号可以让 f 变大也可以变小, 不是极值} \\
\implies \text{极值点必须 } f'(x_0) = 0 \text{ (驻点 stationary point)} \\
\text{2. 二阶条件: 驻点处 } f(x_0 + h) - f(x_0) \approx \frac{1}{2} f''(x_0) h^2 \text{, } f'' > 0 \text{ 极小, } f'' < 0 \text{ 极大} \\ \quad \\
\text{多元: } f(x_0 + h) = f(x_0) + \nabla f^T h + \frac{1}{2} h^T H h + \cdots \\
\text{1. 一阶: } \nabla f(x_0) = 0 \\
\text{2. 二阶: } f(x_0 + h) - f(x_0) \approx \frac{1}{2} h^T H h \\
\quad H \text{ 正定 (所有特征值 > 0): 任意方向都上升} \implies \text{极小值} \\
\quad H \text{ 负定 (所有特征值 < 0)} \implies \text{极大值} \\
\quad H \text{ 不定 (特征值有正有负)} \implies \text{鞍点 (saddle point), 沿某些方向上升, 某些方向下降} \\
\quad \text{(把 h 沿特征向量分解: } h^T H h = \sum_i \lambda_i c_i^2 \text{, 所以符号由特征值决定)}
$$

数学基础: [正定矩阵](linear_algebra.ipynb#positive-definite) · [对称矩阵的特征分解](linear_algebra.ipynb#symmetric-matrix)

In [ ]:
# f(x, y) = x^3 - 3x + y^2
f_crit = lambda x, y: x**3 - 3 * x + y**2
# grad = (3x^2 - 3, 2y) = 0  =>  (1, 0), (-1, 0)
# H = [[6x, 0], [0, 2]]
for point in [(1.0, 0.0), (-1.0, 0.0)]:
    H = np.array([[6 * point[0], 0], [0, 2]])
    eig = np.linalg.eigvalsh(H)
    kind = '极小值' if np.all(eig > 0) else ('极大值' if np.all(eig < 0) else '鞍点')
    print(f'驻点 {point}: Hessian 特征值 = {eig} -> {kind}')

xs, ys = np.meshgrid(np.linspace(-2.2, 2.2, 300), np.linspace(-2, 2, 300))
plt.figure(figsize=(4, 3))
plt.contour(xs, ys, f_crit(xs, ys), levels=30)
plt.plot(1, 0, 'go', label='minimum (1, 0)')
plt.plot(-1, 0, 'rx', markersize=10, label='saddle (-1, 0)')
plt.title('f = x^3 - 3x + y^2')
plot_nicely()
plt.show()

<a id="lagrange"></a>
$$
\text{拉格朗日乘子法 (Lagrange Multiplier): 带等式约束的极值} \\
\min f(x, y) \quad \text{s.t.} \quad g(x, y) = c \\ \quad \\
\text{推导: 沿约束曲线 } r(t) \text{ 移动 (始终满足 } g = c \text{)} \\
1. \ \text{g 在曲线上不变: } \frac{d}{dt} g(r(t)) = \nabla g \cdot r'(t) = 0 \implies \nabla g \perp \text{曲线切向} \\
2. \ \text{在极值点, f 沿曲线的变化率为 0: } \frac{d}{dt} f(r(t)) = \nabla f \cdot r'(t) = 0 \implies \nabla f \perp \text{曲线切向} \\
\text{二维平面里垂直于同一个方向的两个向量一定平行:} \\
\nabla f = \lambda \nabla g \quad (\text{几何上: f 的等高线和约束曲线相切}) \\ \quad \\
\text{写成拉格朗日函数 (Lagrangian): } L(x, y, \lambda) = f(x, y) - \lambda (g(x, y) - c) \\
\frac{\partial L}{\partial x} = 0, \ \frac{\partial L}{\partial y} = 0 \iff \nabla f = \lambda \nabla g, \qquad \frac{\partial L}{\partial \lambda} = 0 \iff g = c \\ \quad \\
\lambda \text{ 的意义 (影子价格 shadow price): 约束放宽一点, 最优值变化多少} \\
f^*(c) = f(x^*(c)) \implies \frac{d f^*}{dc} = \nabla f \cdot \frac{dx^*}{dc} = \lambda \nabla g \cdot \frac{dx^*}{dc} = \lambda \frac{d g(x^*(c))}{dc} = \lambda \frac{dc}{dc} = \lambda \\ \quad \\
\text{例: } \min x^2 + y^2 \ \text{ s.t. } x + 2y = 4 \\
L = x^2 + y^2 - \lambda(x + 2y - 4) \\
\frac{\partial L}{\partial x} = 2x - \lambda = 0, \quad \frac{\partial L}{\partial y} = 2y - 2\lambda = 0 \implies y = 2x \\
x + 4x = 4 \implies x = 0.8, \ y = 1.6, \ \lambda = 1.6, \ f^* = 3.2 \\
\text{(不等式约束的推广是 KKT 条件, 见建模的非线性规划)}
$$

In [ ]:
from scipy.optimize import minimize

# 拉格朗日条件是线性方程组: [2 0 -1; 0 2 -2; 1 2 0] [x y lam]^T = [0 0 4]^T
A_lag = np.array([[2, 0, -1], [0, 2, -2], [1, 2, 0]], dtype=float)
x_l, y_l, lam = np.linalg.solve(A_lag, [0, 0, 4])
print(f'拉格朗日: x = {x_l:.4f}, y = {y_l:.4f}, lambda = {lam:.4f}, f* = {x_l**2 + y_l**2:.4f}')

res = minimize(lambda q: q[0]**2 + q[1]**2, x0=[0, 0],
               constraints={'type': 'eq', 'fun': lambda q: q[0] + 2 * q[1] - 4})
print(f'scipy minimize: x = {res.x}, f* = {res.fun:.4f}')

# 验证 lambda = df*/dc: 一般的 c, x = c/5, y = 2c/5, f*(c) = c^2 / 5
f_star = lambda c: c**2 / 5
print(f'df*/dc (c=4) = {(f_star(4 + 1e-6) - f_star(4 - 1e-6)) / 2e-6:.4f} == lambda = {lam:.4f}')

# 在最优点, grad f 和 grad g 平行
print(f'grad f = {np.array([2 * x_l, 2 * y_l])}, lambda * grad g = {lam * np.array([1, 2])}')

xs, ys = np.meshgrid(np.linspace(-1, 3, 200), np.linspace(-1, 3, 200))
plt.figure(figsize=(4, 3))
plt.contour(xs, ys, xs**2 + ys**2, levels=[0.5, 1.5, 3.2, 5, 8], cmap='viridis')
line_x = np.linspace(-1, 3, 50)
plt.plot(line_x, (4 - line_x) / 2, 'r', label='x + 2y = 4')
plt.plot(x_l, y_l, 'ko', label='optimum (tangent point)')
plt.gca().set_aspect('equal')
plot_nicely()
plt.show()

<a id="gradient-descent"></a>
$$
\text{梯度下降 (Gradient Descent): } x_{k+1} = x_k - \eta \nabla f(x_k) \quad (\eta \text{ 学习率 learning rate}) \\ \quad \\
\text{推导: 一阶泰勒 } f(x - \eta \nabla f) \approx f(x) - \eta \nabla f^T \nabla f = f(x) - \eta |\nabla f|^2 < f(x) \\
\text{只要 } \eta \text{ 足够小, 每一步都会让 f 下降; 并且由方向导数的结论, } -\nabla f \text{ 是下降最快的方向} \\ \quad \\
\text{学习率的上限: 以二次函数为例 } f(x) = \frac{1}{2} x^T A x - b^T x \ (A \text{ 对称正定}) \\
\nabla f = A x - b, \quad \text{最小值点 } x^* = A^{-1} b \\
\text{误差 } e_k = x_k - x^* \text{: } e_{k+1} = x_k - \eta (A x_k - b) - x^* = e_k - \eta A e_k = (I - \eta A) e_k \\
\text{把 } e_k \text{ 沿 A 的特征向量分解, 每个分量每一步乘以 } (1 - \eta \lambda_i) \\
\text{收敛} \iff |1 - \eta \lambda_i| < 1 \ \forall i \iff 0 < \eta < \frac{2}{\lambda_{\max}} \\
\text{收敛速度由最慢的分量决定: } \lambda_{\max} / \lambda_{\min} \text{ (条件数) 越大, 等高线越扁, 越容易 "之" 字形震荡}
$$

In [ ]:
A_gd = np.array([[10.0, 0.0], [0.0, 1.0]])
b_gd = np.array([1.0, 1.0])
x_star = np.linalg.solve(A_gd, b_gd)
f_quad = lambda x, y: 0.5 * (10 * x**2 + y**2) - x - y
print(f'最优点 x* = {x_star}, lambda_max = 10, 学习率上限 2/10 = 0.2')

def gradient_descent(eta, x0, steps=40):
    path = [np.array(x0, dtype=float)]
    for _ in range(steps):
        path.append(path[-1] - eta * (A_gd @ path[-1] - b_gd))
    return np.array(path)

xs, ys = np.meshgrid(np.linspace(-1, 1.2, 200), np.linspace(-0.5, 2.5, 200))
plt.figure(figsize=(4, 3))
plt.contour(xs, ys, f_quad(xs, ys), levels=20, alpha=0.5)
for eta in [0.02, 0.1, 0.19, 0.205]:
    path = gradient_descent(eta, [-0.8, 2.2])
    print(f'eta = {eta:<6}: 40 步后误差 = {np.linalg.norm(path[-1] - x_star):.3e}, '
          f'|1 - eta*lambda| = {np.abs(1 - eta * np.array([10, 1]))}')
    plt.plot(path[:15, 0], path[:15, 1], '.-', label=f'eta={eta}')
plt.xlim(-1, 1.2)
plt.ylim(-0.5, 2.5)
plot_nicely()
plt.show()

<a id="newton"></a>
$$
\text{牛顿法 (Newton's Method)} \\ \quad \\
\text{1. 求根 } f(x) = 0 \text{: 用切线 (一阶泰勒) 近似 f, 求切线的零点} \\
f(x) \approx f(x_k) + f'(x_k)(x - x_k) = 0 \implies x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)} \\ \quad \\
\text{二次收敛 (Quadratic Convergence): 设根为 } x^* \text{, 误差 } e_k = x_k - x^* \text{, 在 } x_k \text{ 处展开} \\
0 = f(x^*) = f(x_k) - f'(x_k) e_k + \frac{1}{2} f''(\xi) e_k^2 \implies \frac{f(x_k)}{f'(x_k)} = e_k - \frac{f''(\xi)}{2 f'(x_k)} e_k^2 \\
e_{k+1} = x_{k+1} - x^* = e_k - \frac{f(x_k)}{f'(x_k)} = \frac{f''(\xi)}{2 f'(x_k)} e_k^2 \\
\text{误差每一步大约平方一次, 有效数字位数每一步翻倍 (二分法每步只减半)} \\ \quad \\
\text{2. 求极值: 对 } f'(x) = 0 \text{ 用牛顿法, } x_{k+1} = x_k - \frac{f'(x_k)}{f''(x_k)} \\
\text{多元: 对二阶泰勒 } f(x + h) \approx f + \nabla f^T h + \frac{1}{2} h^T H h \text{ 关于 h 求梯度令为 0: } \nabla f + H h = 0 \\
x_{k+1} = x_k - H^{-1} \nabla f(x_k) \\
\text{(对二次函数一步就到最优点, 和梯度下降对比: 用了曲率信息, 但要算 Hessian 和解线性方程)}
$$

In [ ]:
from scipy.optimize import newton as scipy_newton

# 求 sqrt(2): f(x) = x^2 - 2
f_root = lambda x: x**2 - 2
df_root = lambda x: 2 * x

x_k = 1.0
print('牛顿法:')
for k in range(6):
    print(f'  k = {k}: x = {x_k:.16f}, 误差 = {abs(x_k - np.sqrt(2)):.2e}')
    x_k = x_k - f_root(x_k) / df_root(x_k)

lo, hi = 1.0, 2.0
print('二分法 (Bisection):')
for k in range(6):
    mid = (lo + hi) / 2
    print(f'  k = {k}: x = {mid:.16f}, 误差 = {abs(mid - np.sqrt(2)):.2e}')
    lo, hi = (mid, hi) if f_root(mid) < 0 else (lo, mid)
print(f'scipy newton = {scipy_newton(f_root, 1.0, fprime=df_root):.16f}')

# 多元牛顿法求极值: 二次函数一步到位
x0 = np.array([-0.8, 2.2])
x1 = x0 - np.linalg.solve(A_gd, A_gd @ x0 - b_gd)
print(f'二次函数牛顿法一步: {x1}, == x* ? {np.allclose(x1, x_star)}')

<a id="integral"></a>
$$
\text{定积分 (Definite Integral): 黎曼和 (Riemann Sum) 的极限} \\
\int_a^b f(x) \, dx = \lim_{n \to \infty} \sum_{i=1}^{n} f(\xi_i) \Delta x, \quad \Delta x = \frac{b - a}{n}, \ \xi_i \in [x_{i-1}, x_i] \\ \quad \\
\text{微积分基本定理 (Fundamental Theorem of Calculus): 积分和导数互为逆运算} \\
\text{令 } F(x) = \int_a^x f(t) \, dt \text{ (面积函数), 则} \\
F(x + h) - F(x) = \int_x^{x+h} f(t) \, dt \approx f(x) \cdot h \implies F'(x) = \lim_{h \to 0} \frac{F(x+h) - F(x)}{h} = f(x) \\
\text{若 G 是 f 的任意一个原函数 } (G' = f) \text{, 则 } (F - G)' = 0 \text{, F - G 是常数, 所以} \\
\int_a^b f(x) \, dx = F(b) - F(a) = G(b) - G(a) \quad (\text{牛顿-莱布尼茨公式}) \\ \quad \\
\text{积分技巧都是求导法则反过来:} \\
\text{分部积分 (Integration by Parts) ← 乘积法则: } (uv)' = u'v + uv' \implies \int u \, dv = uv - \int v \, du \\
\text{换元积分 (Substitution) ← 链式法则: } \frac{d}{dx} G(g(x)) = G'(g(x)) g'(x) \implies \int f(g(x)) g'(x) \, dx = \int f(u) \, du \\ \quad \\
\text{例: } \int_0^1 x e^x \, dx \text{, 取 } u = x, \ dv = e^x dx \text{: } = [x e^x]_0^1 - \int_0^1 e^x \, dx = e - (e - 1) = 1 \\
\text{例: } \int_0^{\sqrt{\pi}} 2x \cos(x^2) \, dx \text{, 令 } u = x^2 \text{: } = \int_0^{\pi} \cos u \, du = \sin\pi - \sin 0 = 0
$$

In [ ]:
f_int = lambda x: x * np.exp(x)
for n in [4, 16, 64, 256, 1024]:
    x_edges = np.linspace(0, 1, n + 1)
    dx = 1 / n
    left = np.sum(f_int(x_edges[:-1])) * dx
    right = np.sum(f_int(x_edges[1:])) * dx
    mid = np.sum(f_int((x_edges[:-1] + x_edges[1:]) / 2)) * dx
    print(f'n = {n:>4}: 左黎曼和 = {left:.6f}, 右黎曼和 = {right:.6f}, 中点 = {mid:.6f}')
print(f'分部积分结果 = 1, scipy quad = {quad(f_int, 0, 1)[0]:.6f}')
print(f'换元积分: quad = {quad(lambda x: 2 * x * np.cos(x**2), 0, np.sqrt(np.pi))[0]:.2e} (== 0)')

# 基本定理: 面积函数的导数 = 被积函数
F_area = lambda x: quad(np.cos, 0, x)[0]
for x0 in [0.5, 1.0, 2.0]:
    print(f'x = {x0}: dF/dx = {(F_area(x0 + 1e-5) - F_area(x0 - 1e-5)) / 2e-5:.6f}, cos(x) = {np.cos(x0):.6f}')

n = 8
x_edges = np.linspace(0, 1, n + 1)
x_fine = np.linspace(0, 1, 200)
plt.figure(figsize=(4, 3))
plt.plot(x_fine, f_int(x_fine), 'k', label='f(x) = x e^x')
plt.bar(x_edges[:-1], f_int(x_edges[:-1]), width=1 / n, align='edge', alpha=0.4, edgecolor='b', label='left Riemann sum')
plot_nicely()
plt.show()

<a id="numerical-integral"></a>
$$
\text{数值积分: 在每个小区间上用多项式近似 f, 再对多项式积分} \\ \quad \\
\text{1. 梯形法 (Trapezoidal Rule): 每个区间用直线 (1 次多项式) 连接两个端点} \\
\int_{x_i}^{x_{i+1}} f \, dx \approx \frac{h}{2} (f_i + f_{i+1}) \implies \int_a^b f \, dx \approx h \left( \frac{f_0}{2} + f_1 + \cdots + f_{n-1} + \frac{f_n}{2} \right) \\
\text{误差 } O(h^2) \text{ (单个区间 } O(h^3) \text{, 共 } n = (b - a)/h \text{ 个区间)} \\ \quad \\
\text{2. 辛普森法 (Simpson's Rule): 每两个区间用一条抛物线 (2 次多项式) 过三个点} \\
\text{推导: 平移到 } [-h, h] \text{, 设 } p(x) = A x^2 + B x + C \\
\int_{-h}^{h} p(x) \, dx = \frac{2 A h^3}{3} + 2 C h \quad (\text{奇函数 Bx 积分为 0}) \\
p(-h) + p(h) = 2 A h^2 + 2C, \quad p(0) = C \implies A h^2 = \frac{p(-h) + p(h)}{2} - p(0) \\
\int_{-h}^{h} p \, dx = \frac{2h}{3}\left(\frac{p(-h) + p(h)}{2} - p(0)\right) + 2h \, p(0) = \frac{h}{3}\left[p(-h) + 4 p(0) + p(h)\right] \\
\int_a^b f \, dx \approx \frac{h}{3} (f_0 + 4 f_1 + 2 f_2 + 4 f_3 + \cdots + 4 f_{n-1} + f_n) \quad (n \text{ 为偶数}) \\
\text{误差 } O(h^4) \text{ (对称性让 3 次项也正好积分为 0, 比预期多一阶)}
$$

In [ ]:
from scipy.integrate import trapezoid, simpson

def trapezoid_manual(f, a, b, n):
    x = np.linspace(a, b, n + 1)
    h = (b - a) / n
    return h * (f(x[0]) / 2 + f(x[1:-1]).sum() + f(x[-1]) / 2)

def simpson_manual(f, a, b, n):
    assert n % 2 == 0
    x = np.linspace(a, b, n + 1)
    h = (b - a) / n
    return h / 3 * (f(x[0]) + 4 * f(x[1:-1:2]).sum() + 2 * f(x[2:-1:2]).sum() + f(x[-1]))

exact = 2.0   # int_0^pi sin x dx
x_check = np.linspace(0, np.pi, 11)
print(f'梯形 手写 == scipy? {np.isclose(trapezoid_manual(np.sin, 0, np.pi, 10), trapezoid(np.sin(x_check), x=x_check))}')
print(f'辛普森 手写 == scipy? {np.isclose(simpson_manual(np.sin, 0, np.pi, 10), simpson(np.sin(x_check), x=x_check))}')

ns = np.array([4, 8, 16, 32, 64, 128])
err_trap = [abs(trapezoid_manual(np.sin, 0, np.pi, n) - exact) for n in ns]
err_simp = [abs(simpson_manual(np.sin, 0, np.pi, n) - exact) for n in ns]
for n, e1, e2 in zip(ns, err_trap, err_simp):
    print(f'n = {n:>3}: 梯形误差 = {e1:.3e}, 辛普森误差 = {e2:.3e}')
print('n 翻倍 (h 减半): 梯形误差约 1/4, 辛普森误差约 1/16')

hs = np.pi / ns
plt.figure(figsize=(4, 3))
plt.loglog(hs, err_trap, 'o-', label='trapezoid')
plt.loglog(hs, err_simp, 's-', label='Simpson')
plt.loglog(hs, 0.2 * hs**2, 'k--', label='slope 2')
plt.loglog(hs, 0.01 * hs**4, 'k:', label='slope 4')
plt.xlabel('h')
plt.ylabel('|error|')
plot_nicely()
plt.show()

<a id="multiple-integral"></a>
$$
\text{重积分与换元 (Change of Variables)} \\
\iint_D f(x, y) \, dx\,dy = \iint_{D'} f(x(u, v), y(u, v)) \, |\det J| \, du\,dv \\
\text{推导: (u, v) 平面上一个 } du \times dv \text{ 的小矩形, 在 (x, y) 平面上变成两条边为 } J e_1 du, \ J e_2 dv \text{ 的平行四边形} \\
\text{平行四边形面积 = 两条边组成的矩阵的行列式绝对值 = } |\det J| \, du \, dv \\ \quad \\
\text{例: 高斯积分 (Gaussian Integral) } I = \int_{-\infty}^{\infty} e^{-x^2} dx \text{, 直接找不到原函数, 先平方变成二重积分} \\
I^2 = \int_{-\infty}^{\infty} e^{-x^2} dx \int_{-\infty}^{\infty} e^{-y^2} dy = \iint_{\mathbb{R}^2} e^{-(x^2 + y^2)} dx\,dy \\
\text{换成极坐标, } |\det J| = r \text{:} \\
I^2 = \int_0^{2\pi} \int_0^{\infty} e^{-r^2} \, r \, dr \, d\theta = 2\pi \left[-\frac{1}{2} e^{-r^2}\right]_0^{\infty} = \pi \implies I = \sqrt{\pi} \\
\text{(多出来的 r 让 } e^{-r^2} r \text{ 有了原函数, 这是换元的关键)} \\ \quad \\
\text{推论: 正态分布 PDF 的归一化常数, 令 } x = \frac{t - \mu}{\sqrt{2}\sigma} \text{:} \\
\int_{-\infty}^{\infty} e^{-\frac{(t - \mu)^2}{2\sigma^2}} dt = \sqrt{2}\sigma \int_{-\infty}^{\infty} e^{-x^2} dx = \sqrt{2\pi}\sigma \implies f(t) = \frac{1}{\sqrt{2\pi}\sigma} e^{-\frac{(t - \mu)^2}{2\sigma^2}} \text{ 积分为 1}
$$

数学基础: [正态分布](statistics.ipynb#normal-distribution)

In [ ]:
from scipy.integrate import dblquad

I, _ = quad(lambda x: np.exp(-x**2), -np.inf, np.inf)
print(f'高斯积分 = {I:.10f}, sqrt(pi) = {np.sqrt(np.pi):.10f}')

# 直角坐标的二重积分 vs 极坐标 (多了 r)
cart, _ = dblquad(lambda y, x: np.exp(-(x**2 + y**2)), -8, 8, -8, 8)
pol, _ = dblquad(lambda r, th: np.exp(-r**2) * r, 0, 2 * np.pi, 0, np.inf)
pol_wrong, _ = dblquad(lambda r, th: np.exp(-r**2), 0, 2 * np.pi, 0, np.inf)
print(f'直角坐标 = {cart:.8f}, 极坐标 (带 r) = {pol:.8f}, 忘记 r = {pol_wrong:.8f}, pi = {np.pi:.8f}')

# 蒙特卡洛验证面积元: 单位圆内均匀撒点, r 的分布密度应该正比于 r (面积元 r dr dθ)
rng = np.random.default_rng(0)
pts = rng.uniform(-1, 1, size=(200000, 2))
r_inside = np.linalg.norm(pts, axis=1)
r_inside = r_inside[r_inside < 1]
plt.figure(figsize=(4, 3))
plt.hist(r_inside, bins=50, density=True, alpha=0.6, label='r of uniform points in disk')
plt.plot([0, 1], [0, 2], 'r', label='density = 2r')
plt.xlabel('r')
plot_nicely()
plt.show()

sigma = 1.7
print(f'正态 PDF (sigma = {sigma}) 积分 = {quad(lambda t: np.exp(-t**2 / (2 * sigma**2)) / (np.sqrt(2 * np.pi) * sigma), -np.inf, np.inf)[0]:.10f}')

<a id="ode"></a>
$$
\text{常微分方程 (Ordinary Differential Equation, ODE): 含有未知函数及其导数的方程} \\ \quad \\
\text{从导数的定义建立 ODE: 描述 "变化率" 的规律} \\
\text{例 1: 人口 / 细菌 / 存款, 单位时间的增长量正比于当前数量} \\
y(t + \Delta t) - y(t) \approx k \, y(t) \, \Delta t \implies \frac{y(t + \Delta t) - y(t)}{\Delta t} \approx k \, y(t) \xrightarrow{\Delta t \to 0} \frac{dy}{dt} = k y \\ \quad \\
\text{例 2: 牛顿冷却定律 (Newton's Law of Cooling), 降温速度正比于和环境的温差} \\
\frac{dT}{dt} = -k (T - T_{env}) \\ \quad \\
\text{例 3: 牛顿第二定律, 速度是位置的导数, 加速度是速度的导数} \\
F = m a = m \frac{dv}{dt} = m \frac{d^2 x}{dt^2} \text{, 弹簧 } F = -kx \implies \frac{d^2 x}{dt^2} = -\frac{k}{m} x \text{ (二阶 ODE)} \\ \quad \\
\text{阶 (Order): 出现的最高阶导数; 解 ODE 需要初始条件 (Initial Condition), n 阶需要 n 个}
$$

<a id="ode-separable"></a>
$$
\text{分离变量法 (Separation of Variables): } \frac{dy}{dt} = h(t) \, g(y) \\
\text{写成 } \frac{1}{g(y)} \frac{dy}{dt} = h(t) \text{, 设 G 是 } \frac{1}{g} \text{ 的原函数, 由链式法则} \\
\frac{d}{dt} G(y(t)) = G'(y) \frac{dy}{dt} = \frac{1}{g(y)} \frac{dy}{dt} = h(t) \\
\text{两边对 t 积分: } G(y) = \int h(t) \, dt + C \quad (\text{形式上就是 } \int \frac{dy}{g(y)} = \int h(t) \, dt) \\ \quad \\
\text{例 1: } \frac{dy}{dt} = k y \implies \int \frac{dy}{y} = \int k \, dt \implies \ln|y| = kt + C \implies y = y_0 e^{kt} \\ \quad \\
\text{例 2: 牛顿冷却, 令 } u = T - T_{env} \text{, } \frac{du}{dt} = \frac{dT}{dt} = -k u \implies u = u_0 e^{-kt} \\
T(t) = T_{env} + (T_0 - T_{env}) e^{-kt} \quad (\text{RC 电路充放电是完全相同的方程})
$$

In [ ]:
k, T_env, T0 = 0.1, 20.0, 90.0
t_eval = np.linspace(0, 40, 200)

T_analytic = T_env + (T0 - T_env) * np.exp(-k * t_eval)
sol = solve_ivp(lambda t, T: -k * (T - T_env), [0, 40], [T0], t_eval=t_eval, rtol=1e-10, atol=1e-10)
print(f'分离变量解 == solve_ivp ? {np.allclose(T_analytic, sol.y[0], atol=1e-6)}')
print(f'10 分钟后温度 = {T_env + (T0 - T_env) * np.exp(-k * 10):.4f}')
# 反过来: 降到 40 度要多久? 40 = 20 + 70 e^{-kt} => t = ln(70 / 20) / k
print(f'降到 40 度需要 t = ln(70/20) / k = {np.log(70 / 20) / k:.4f}')

plt.figure(figsize=(4, 3))
plt.plot(t_eval, T_analytic, label='analytic')
plt.plot(t_eval[::10], sol.y[0][::10], 'o', label='solve_ivp')
plt.axhline(T_env, color='gray', linestyle='--', label='T_env')
plt.xlabel('t')
plt.ylabel('T')
plt.title("Newton's law of cooling")
plot_nicely()
plt.show()

<a id="ode-linear"></a>
$$
\text{一阶线性 ODE: } \frac{dy}{dt} + p(t) \, y = q(t) \text{ (q 不为 0 时不能分离变量)} \\ \quad \\
\text{积分因子法 (Integrating Factor): 想把左边凑成某个乘积的导数} \\
\text{乘以 } \mu(t) \text{: } \mu y' + \mu p y \text{, 对比乘积法则 } (\mu y)' = \mu y' + \mu' y \\
\text{只要 } \mu' = \mu p \text{ (这是一个可分离变量的 ODE)} \implies \mu(t) = e^{\int p(t) dt} \\
(\mu y)' = \mu q \implies y = \frac{1}{\mu(t)} \left( \int \mu(t) q(t) \, dt + C \right) \\ \quad \\
\text{例: } y' + 2y = \sin t, \ y(0) = 1 \\
\mu = e^{2t}, \quad (e^{2t} y)' = e^{2t} \sin t \\
\int e^{2t} \sin t \, dt = \frac{e^{2t}(2 \sin t - \cos t)}{5} \quad (\text{分部积分两次, 可以求导验证}) \\
y = \frac{2 \sin t - \cos t}{5} + C e^{-2t}, \quad y(0) = -\frac{1}{5} + C = 1 \implies C = \frac{6}{5} \\
\underbrace{\frac{6}{5} e^{-2t}}_{\text{暂态 transient, 衰减}} + \underbrace{\frac{2 \sin t - \cos t}{5}}_{\text{稳态 steady state, 跟着输入振荡}} \quad (\text{RC 电路接交流电就是这个形式})
$$

In [ ]:
y_exact = lambda t: (2 * np.sin(t) - np.cos(t)) / 5 + 6 / 5 * np.exp(-2 * t)
ode_rhs = lambda t, y: np.sin(t) - 2 * y      # y' = q - p y

# 验证: 积分 e^{2t} sin t 的原函数 (求导回去)
G_anti = lambda t: np.exp(2 * t) * (2 * np.sin(t) - np.cos(t)) / 5
t0 = 0.9
print(f'原函数求导 = {(G_anti(t0 + 1e-6) - G_anti(t0 - 1e-6)) / 2e-6:.6f}, e^(2t) sin t = {np.exp(2 * t0) * np.sin(t0):.6f}')

t_eval = np.linspace(0, 10, 300)
sol = solve_ivp(ode_rhs, [0, 10], [1.0], t_eval=t_eval, rtol=1e-10, atol=1e-12)
print(f'积分因子解 == solve_ivp ? {np.allclose(sol.y[0], y_exact(t_eval), atol=1e-8)}')

plt.figure(figsize=(4, 3))
plt.plot(t_eval, y_exact(t_eval), label='y(t)')
plt.plot(t_eval, (2 * np.sin(t_eval) - np.cos(t_eval)) / 5, '--', label='steady state')
plt.plot(t_eval, 6 / 5 * np.exp(-2 * t_eval), ':', label='transient')
plt.xlabel('t')
plot_nicely()
plt.show()

<a id="euler-rk4"></a>
$$
\text{数值解 ODE: } y' = f(t, y), \ y(t_0) = y_0 \text{, 步长 h} \\ \quad \\
\text{1. 欧拉法 (Euler Method): 直接用导数的定义 / 一阶泰勒} \\
y(t + h) = y(t) + h \, y'(t) + O(h^2) = y(t) + h f(t, y) + O(h^2) \\
y_{n+1} = y_n + h f(t_n, y_n) \\
\text{每一步局部误差 } O(h^2) \text{, 走 } (T - t_0)/h \text{ 步累积成全局误差 } O(h) \\ \quad \\
\text{2. 改进: 把 ODE 两边积分, 变成数值积分问题} \\
y(t_{n+1}) = y(t_n) + \int_{t_n}^{t_{n+1}} f(s, y(s)) \, ds \\
\text{欧拉法 = 用左端点的矩形近似这个积分} \\
\text{用梯形法 → Heun 法 (改进欧拉): } k_1 = f(t_n, y_n), \ k_2 = f(t_n + h, y_n + h k_1), \ y_{n+1} = y_n + \frac{h}{2}(k_1 + k_2) \text{, 全局 } O(h^2) \\ \quad \\
\text{3. 四阶龙格-库塔 (RK4) ← 用辛普森法近似这个积分 } \frac{h}{6}[f(t_n) + 4 f(t_n + \tfrac{h}{2}) + f(t_n + h)] \\
\text{中点的 } y(t_n + \tfrac{h}{2}) \text{ 未知, 用两种方法估计, 中间的 4 拆成 2 + 2:} \\
k_1 = f(t_n, y_n) \quad (\text{起点斜率}) \\
k_2 = f(t_n + \tfrac{h}{2}, \ y_n + \tfrac{h}{2} k_1) \quad (\text{用 } k_1 \text{ 走半步, 估计中点斜率}) \\
k_3 = f(t_n + \tfrac{h}{2}, \ y_n + \tfrac{h}{2} k_2) \quad (\text{用 } k_2 \text{ 再估计一次中点斜率}) \\
k_4 = f(t_n + h, \ y_n + h k_3) \quad (\text{用 } k_3 \text{ 走一整步, 估计终点斜率}) \\
y_{n+1} = y_n + \frac{h}{6}(k_1 + 2 k_2 + 2 k_3 + k_4) \\
\text{全局误差 } O(h^4) \text{ (和泰勒展开逐项对比可以严格证明, 这里用数值验证)}
$$

In [ ]:
def euler(f, t0, y0, h, n_steps):
    t, y = t0, np.array(y0, dtype=float)
    ts, ys = [t], [y.copy()]
    for _ in range(n_steps):
        y = y + h * f(t, y)
        t = t + h
        ts.append(t); ys.append(y.copy())
    return np.array(ts), np.array(ys)

def heun(f, t0, y0, h, n_steps):
    t, y = t0, np.array(y0, dtype=float)
    ts, ys = [t], [y.copy()]
    for _ in range(n_steps):
        k1 = f(t, y)
        k2 = f(t + h, y + h * k1)
        y = y + h / 2 * (k1 + k2)
        t = t + h
        ts.append(t); ys.append(y.copy())
    return np.array(ts), np.array(ys)

def rk4(f, t0, y0, h, n_steps):
    t, y = t0, np.array(y0, dtype=float)
    ts, ys = [t], [y.copy()]
    for _ in range(n_steps):
        k1 = f(t, y)
        k2 = f(t + h / 2, y + h / 2 * k1)
        k3 = f(t + h / 2, y + h / 2 * k2)
        k4 = f(t + h, y + h * k3)
        y = y + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        t = t + h
        ts.append(t); ys.append(y.copy())
    return np.array(ts), np.array(ys)

T_end = 5.0
results = {'Euler': [], 'Heun': [], 'RK4': []}
hs = T_end / np.array([10, 20, 40, 80, 160, 320])
for h in hs:
    n_steps = int(round(T_end / h))
    for name, method in [('Euler', euler), ('Heun', heun), ('RK4', rk4)]:
        _, ys = method(ode_rhs, 0.0, [1.0], h, n_steps)
        results[name].append(abs(ys[-1, 0] - y_exact(T_end)))

for name, errs in results.items():
    orders = np.log2(np.array(errs[:-1]) / np.array(errs[1:]))
    print(f'{name:<5}: 误差 = {np.array(errs)}, 估计阶数 = {np.round(orders, 2)}')

plt.figure(figsize=(4, 3))
for name, errs in results.items():
    plt.loglog(hs, errs, 'o-', label=name)
plt.xlabel('h')
plt.ylabel('|error| at t = 5')
plot_nicely()
plt.show()

<a id="ode-system"></a>
$$
\text{高阶 ODE → 一阶 ODE 方程组: 把导数当成新的未知函数} \\
x'' = -\omega^2 x \text{, 令 } v = x' \implies \begin{cases} x' = v \\ v' = -\omega^2 x \end{cases} \implies \frac{d}{dt}\begin{bmatrix} x \\ v \end{bmatrix} = \begin{bmatrix} 0 & 1 \\ -\omega^2 & 0 \end{bmatrix} \begin{bmatrix} x \\ v \end{bmatrix} \\
\text{这样 Euler / RK4 / solve_ivp 都可以直接用 (y 变成向量)} \\ \quad \\
\text{常系数线性 ODE 的解析解: 猜 } x = e^{rt} \text{ (因为指数函数求导后形状不变)} \\
r^2 e^{rt} = -\omega^2 e^{rt} \implies r^2 + \omega^2 = 0 \ (\text{特征方程 characteristic equation}) \implies r = \pm i\omega \\
x = c_1 e^{i\omega t} + c_2 e^{-i\omega t} \xrightarrow{\text{欧拉公式}} x = A \cos\omega t + B \sin\omega t \\
\text{初始条件 } x(0) = x_0, \ v(0) = v_0 \implies A = x_0, \ B = \frac{v_0}{\omega} \\ \quad \\
\text{守恒量: 能量 } E = \frac{1}{2} v^2 + \frac{1}{2} \omega^2 x^2 \text{, } \frac{dE}{dt} = v v' + \omega^2 x x' = v(-\omega^2 x) + \omega^2 x v = 0 \\
\text{欧拉法每一步: } E_{n+1} = (1 + h^2 \omega^2) E_n \text{, 能量会指数增长 (轨道向外螺旋)} \\
\text{(和 ARIMA 的特征多项式是同一个思想: 差分方程猜 } \lambda^t \text{, 微分方程猜 } e^{rt} \text{)}
$$

数学基础: [特征值与特征向量](linear_algebra.ipynb#eigen) · 应用: [时间序列的特征多项式与单位根](../modeling/time_series_arima_model/time_series_arima_model.ipynb#unit-root)

In [ ]:
omega = 2.0
x0_osc, v0_osc = 1.0, 0.0
osc = lambda t, s: np.array([s[1], -omega**2 * s[0]])
energy = lambda s: 0.5 * s[..., 1]**2 + 0.5 * omega**2 * s[..., 0]**2

h, n_steps = 0.05, 200
ts, ys_euler = euler(osc, 0.0, [x0_osc, v0_osc], h, n_steps)
_, ys_rk4 = rk4(osc, 0.0, [x0_osc, v0_osc], h, n_steps)
x_true = x0_osc * np.cos(omega * ts) + v0_osc / omega * np.sin(omega * ts)
sol = solve_ivp(osc, [0, ts[-1]], [x0_osc, v0_osc], t_eval=ts, rtol=1e-10, atol=1e-10)

print(f'RK4 最大误差 = {np.max(np.abs(ys_rk4[:, 0] - x_true)):.2e}, solve_ivp 最大误差 = {np.max(np.abs(sol.y[0] - x_true)):.2e}')
print(f'初始能量 = {energy(ys_euler[0]):.4f}, Euler 最终能量 = {energy(ys_euler[-1]):.4f}, '
      f'理论 (1 + h^2 w^2)^n E0 = {(1 + h**2 * omega**2)**n_steps * energy(ys_euler[0]):.4f}, RK4 最终能量 = {energy(ys_rk4[-1]):.4f}')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(ts, x_true, 'k', label='analytic')
axs[0].plot(ts, ys_euler[:, 0], '--', label='Euler')
axs[0].plot(ts, ys_rk4[:, 0], ':', label='RK4')
axs[0].set_xlabel('t')
axs[0].set_ylabel('x')
axs[1].plot(ys_euler[:, 0], ys_euler[:, 1], '--', label='Euler (spirals out)')
axs[1].plot(ys_rk4[:, 0], ys_rk4[:, 1], label='RK4')
axs[1].set_xlabel('x')
axs[1].set_ylabel('v')
axs[1].set_title('phase space')
axs[1].set_aspect('equal')
plot_nicely()
plt.show()